# 06 Avoiding UDFs and Unnecessary Shuffles

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Rewrite slow code into fast code: replace Python UDFs with built-in functions (regex, JSON, lookups, dates, arrays), and remove shuffles you don't need (redundant <code>repartition</code>, <code>distinct</code>, <code>orderBy</code>, and join-back patterns that a window function can replace). Measure every rewrite.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
UDFs and extra shuffles are the two most common reasons PySpark code is slower than it should be, and both are fixed in code, not with bigger clusters. Rewriting a UDF into built-ins is a typical interview exercise ("how would you make this faster?"), and so is spotting a redundant shuffle in a plan.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Replacement guide</b><br>
| UDF that... | Built-in replacement |<br>|---|---|<br>| Parses text with regex | <code>regexp_extract</code>, <code>regexp_replace</code>, <code>split</code>, <code>rlike</code> |<br>| Parses JSON strings | <code>from_json</code>, <code>get_json_object</code>, <code>:</code> path syntax in Databricks SQL |<br>| Looks up values in a Python dict | <code>create_map</code> + <code>element_at</code>, or a broadcast join |<br>| Does date logic | <code>date_add</code>, <code>datediff</code>, <code>date_trunc</code>, <code>dayofweek</code>, <code>make_date</code> |<br>| Loops over a list | Higher-order functions: <code>transform</code>, <code>filter</code>, <code>aggregate</code>, <code>exists</code> |<br>| Implements if/else | <code>when</code> / <code>otherwise</code>, <code>CASE WHEN</code> |<br>| Hashes or encodes | <code>sha2</code>, <code>md5</code>, <code>base64</code>, <code>hex</code> |<br>| Needs a Python library | pandas UDF (vectorized), or <code>mapInPandas</code> |<br><br>
| Unnecessary shuffle | Fix |<br>|---|---|<br>| <code>repartition</code> before a <code>groupBy</code>/<code>join</code> on another key | Remove it |<br>| <code>orderBy</code> in the middle of a pipeline | Sort only at the end, if at all |<br>| <code>distinct</code> before an aggregation that ignores duplicates | Remove it, or use <code>countDistinct</code> |<br>| <code>groupBy</code> + join back to add group totals | Window function over the same key (one shuffle) |<br>| Several aggregations of the same data | One <code>groupBy().agg(...)</code> with all metrics |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A Silver job had five Python UDFs: one parsed a product code with regex, one looked up a region in a dict, one computed a fiscal week, one parsed a JSON attributes string, and one normalized phone numbers. Each forced rows through Python. Rewriting all five with built-ins took an afternoon, and the job went from 48 minutes to 7, with Photon now accelerating the whole stage.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an orders DataFrame with a product code, a JSON attributes string, a region code, a date and an array of item prices.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A sample of 3 rows.
</div>

In [0]:
import time, json
from pyspark.sql import functions as F
from pyspark.sql.functions import udf

orders = spark.range(0, 1_000_000, 1, numPartitions=8).select(
    F.col("id").alias("order_id"),
    F.concat(F.lit("SKU-"), F.element_at(F.array(*[F.lit(c) for c in ["IN", "UK", "AE"]]), (F.col("id") % 3 + 1).cast("int")),
             F.lit("-"), F.lpad((F.col("id") % 99_999).cast("string"), 5, "0")).alias("product_code"),
    F.concat(F.lit('{"channel": "'), F.element_at(F.array(F.lit("app"), F.lit("web")), (F.col("id") % 2 + 1).cast("int")),
             F.lit('", "coupon": "C'), (F.col("id") % 10).cast("string"), F.lit('"}')).alias("attributes"),
    (F.col("id") % 4).alias("region_code"),
    F.expr("date_add(date'2024-01-01', cast(id % 365 AS INT))").alias("order_date"),
    F.array(*[(F.col("id") % (k + 7)).cast("double") for k in range(5)]).alias("item_prices"),
)
orders.show(3, truncate=False)

def timed(label, df):
    start = time.time()
    df.write.format("noop").mode("overwrite").save()
    return time.time() - start

def compare(name, udf_df, builtin_df):
    t_udf, t_builtin = timed(name, udf_df), timed(name, builtin_df)
    same = sorted(udf_df.limit(1000).collect()) == sorted(builtin_df.limit(1000).collect())
    print(f"{name:<28} UDF {t_udf:6.2f}s | built-in {t_builtin:6.2f}s | {t_udf / t_builtin:4.1f}x faster | same: {same}")

## 1. Regex parsing

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Extracts the country from the product code with a Python regex UDF, and with <code>regexp_extract</code>.<br><br>
<b>Why it matters</b><br>Text parsing is the most common UDF in the wild, and it's almost always replaceable.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both produce the same countries. The built-in version is several times faster.
</div>

In [0]:
import re

@udf("string")
def country_udf(code):
    m = re.match(r"SKU-([A-Z]{2})-", code or "")
    return m.group(1) if m else None

compare("regex extract",
        orders.select("order_id", country_udf("product_code").alias("country")),
        orders.select("order_id", F.regexp_extract("product_code", r"SKU-([A-Z]{2})-", 1).alias("country")))

## 2. JSON parsing

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the <code>channel</code> from the JSON attributes string with a UDF (<code>json.loads</code>), and with <code>from_json</code> / <code>get_json_object</code>.<br><br>
<b>Why it matters</b><br><code>from_json</code> parses into a typed struct inside the JVM, and Databricks SQL's <code>attributes:channel</code> syntax is even shorter.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Same channels, much faster built-in.
</div>

In [0]:
@udf("string")
def channel_udf(s):
    return json.loads(s).get("channel") if s else None

compare("json parsing",
        orders.select("order_id", channel_udf("attributes").alias("channel")),
        orders.select("order_id", F.from_json("attributes", "channel STRING, coupon STRING")["channel"].alias("channel")))

## 3. Dictionary lookups

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Maps region codes to names with a UDF over a Python dict, then with <code>create_map</code> (a map literal), then with a broadcast join.<br><br>
<b>Why it matters</b><br>Small mappings belong in a map expression or a lookup table. A lookup table is also easier to maintain than code, because business users can update it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All three give the same region names. Both built-in versions are much faster than the UDF.
</div>

In [0]:
REGIONS = {0: "North", 1: "South", 2: "East", 3: "West"}

@udf("string")
def region_udf(code):
    return REGIONS.get(code)

region_map = F.create_map(*[F.lit(x) for kv in REGIONS.items() for x in kv])
regions_df = spark.createDataFrame(list(REGIONS.items()), "region_code BIGINT, region STRING")

compare("dict lookup (map literal)",
        orders.select("order_id", region_udf("region_code").alias("region")),
        orders.select("order_id", region_map[F.col("region_code")].alias("region")))
print("broadcast join version equals map version:",
      sorted(orders.join(F.broadcast(regions_df), "region_code").select("order_id", "region").limit(1000).collect())
      == sorted(orders.select("order_id", region_map[F.col("region_code")].alias("region")).limit(1000).collect()))

## 4. Date logic

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes a fiscal quarter (fiscal year starting in April) with a UDF and with built-ins.<br><br>
<b>Why it matters</b><br>Date arithmetic is fully covered by built-ins (lesson 01_15). Python date code in UDFs also risks time zone surprises.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Same fiscal quarters, faster built-in.
</div>

In [0]:
@udf("string")
def fiscal_quarter_udf(d):
    fy = d.year if d.month >= 4 else d.year - 1
    q = ((d.month - 4) % 12) // 3 + 1
    return f"FY{fy}-Q{q}"

fy = F.when(F.month("order_date") >= 4, F.year("order_date")).otherwise(F.year("order_date") - 1)
fq = (((F.month("order_date") - 4 + 12) % 12) / 3).cast("int") + 1
compare("fiscal quarter",
        orders.select("order_id", fiscal_quarter_udf("order_date").alias("fq")),
        orders.select("order_id", F.concat(F.lit("FY"), fy.cast("string"), F.lit("-Q"), fq.cast("string")).alias("fq")))

## 5. Array logic

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sums the item prices above 3 with a UDF looping over the list, and with higher-order functions.<br><br>
<b>Why it matters</b><br><code>filter</code>, <code>transform</code> and <code>aggregate</code> run list logic inside the JVM, without exploding rows.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Same totals, faster built-in.
</div>

In [0]:
@udf("double")
def sum_over_3_udf(prices):
    return float(sum(p for p in prices if p > 3))

compare("array logic",
        orders.select("order_id", sum_over_3_udf("item_prices").alias("s")),
        orders.select("order_id", F.aggregate(F.filter("item_prices", lambda p: p > 3), F.lit(0.0), lambda acc, p: acc + p).alias("s")))

## 6. Removing unnecessary shuffles

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes each order's share of its region's revenue three ways: <code>groupBy</code> + join back, a window function, and the same with a redundant <code>repartition</code> and <code>orderBy</code> added. Counts the shuffles in each plan.<br><br>
<b>Why it matters</b><br>Adding a group-level value to every row is very common. The join-back version shuffles the data twice, while a window over the same key shuffles once. Redundant <code>repartition</code> and mid-pipeline <code>orderBy</code> add shuffles that change nothing in the result.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Join-back: 2 or more shuffles (unless the totals are broadcast). Window: 1 shuffle. The redundant version: the most shuffles and the slowest timing. All three results match.
</div>

In [0]:
import io, contextlib, re as _re
from pyspark.sql import Window

def shuffles(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return len(_re.findall(r"(?<!Broadcast)Exchange (hash|range|RoundRobin|SinglePartition)", buf.getvalue()))

base = orders.select("order_id", "region_code", (F.col("order_id") % 500).cast("double").alias("amount"))

totals = base.groupBy("region_code").agg(F.sum("amount").alias("region_total"))
join_back = base.join(totals.hint("merge"), "region_code").withColumn("share", F.col("amount") / F.col("region_total"))
window = base.withColumn("share", F.col("amount") / F.sum("amount").over(Window.partitionBy("region_code")))
redundant = base.repartition(64).orderBy("order_id").withColumn("share", F.col("amount") / F.sum("amount").over(Window.partitionBy("region_code")))

for name, df in [("groupBy + join back", join_back), ("window function", window), ("window + redundant shuffles", redundant)]:
    print(f"{name:<30} shuffles: {shuffles(df)} | {timed(name, df):.2f}s")

check = lambda df: sorted((r["order_id"], round(r["share"], 12)) for r in df.select("order_id", "share").limit(2000).collect())
print("same shares:", check(join_back.orderBy("order_id")) == check(window.orderBy("order_id")))

## 7. When you truly need Python: pandas UDF

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes <code>log1p</code> three ways: a classic UDF calling NumPy, a pandas UDF calling NumPy, and the built-in <code>F.log1p</code>.<br><br>
<b>Why it matters</b><br>If a Python library is truly required, a pandas UDF (Arrow batches, vectorized code) is the next best option. But check first: here a built-in exists, and it beats both UDFs by a wide margin.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The built-in is many times faster than either UDF. On Databricks the pandas UDF is usually clearly faster than the classic UDF. At this size, Python worker overhead can make the two UDFs look similar.
</div>

In [0]:
import pandas as pd, numpy as np
from pyspark.sql.functions import pandas_udf

@udf("double")
def log_scale_udf(x):
    return float(np.log1p(x))

@pandas_udf("double")
def log_scale_pandas(x: pd.Series) -> pd.Series:
    return np.log1p(x)

t1 = timed("classic", base.select(log_scale_udf("amount")))
t2 = timed("pandas", base.select(log_scale_pandas("amount")))
t3 = timed("built-in", base.select(F.log1p("amount")))
print(f"classic UDF {t1:.2f}s | pandas UDF {t2:.2f}s | built-in log1p {t3:.2f}s")

## Under the hood

```
built-in:   Scan ─▶ *(1) Project [regexp_extract(...), from_json(...)]      one fused stage, JVM / Photon
UDF:        Scan ─▶ BatchEvalPython [country_udf(...)] ─▶ Project            rows to Python and back

join back:  base ─shuffle─┐                   window:  base ─shuffle─▶ Window(sum) ─▶ Project
            base ─agg─shuffle─┴─ join                    (one shuffle, rows never rejoined)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> built-in expressions are fused into the surrounding code-generated stage.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> every extra `Exchange` writes and reads the whole dataset again.

In [0]:
orders.select(country_udf("product_code")).explain()
orders.select(F.regexp_extract("product_code", r"SKU-([A-Z]{2})-", 1)).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: "there's no built-in for this"</b><br>
Check the functions reference: regex, JSON, maps, arrays, higher-order functions, dates and hashes cover most cases. Combine them.<br><br>
<b>⛔ Problem: rewritten logic gives different results</b><br>Edge cases (nulls, empty strings, time zones) behave differently. Compare UDF and built-in outputs on a sample, as <code>compare</code> does.<br><br>
**⛔ Problem: <code>groupBy</code> then join back to add totals**<br>Use a window function over the same key.<br><br>
**⛔ Problem: <code>repartition</code> and <code>orderBy</code> sprinkled through the pipeline**<br>Remove them unless a later step needs them. Check the shuffle count in <code>explain()</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How would you speed up a PySpark job that uses several Python UDFs?</b><br>
Replace each UDF with built-in functions: regex functions for parsing, <code>from_json</code> for JSON, map expressions or broadcast joins for lookups, date functions, and higher-order functions for arrays. Built-ins run in the JVM or Photon, are optimized and code-generated, and avoid Python serialization. Only if a Python library is truly needed, use a pandas UDF.<br><br>

<b>🎤 2. How do you add a group total to every row efficiently?</b><br>
With a window function partitioned by the group key, for example <code>sum("amount").over(Window.partitionBy("region"))</code>, which needs one shuffle. A <code>groupBy</code> followed by a join back needs more.<br><br>

<b>🎤 3. How do you find unnecessary shuffles?</b><br>
Count the <code>Exchange</code> operators in <code>explain()</code> or the query profile, and ask for each one whether it's required by a later operation. Typical waste: <code>repartition</code> before a <code>groupBy</code> on another key, <code>orderBy</code> in the middle, <code>distinct</code> before an aggregation that already deduplicates, and several separate aggregations of the same data.<br><br>

<b>🎤 4. When is a pandas UDF acceptable?</b><br>
When the logic needs a Python library or numerical code with no built-in equivalent. It's vectorized over Arrow batches, so it's much faster than a row-by-row UDF, though still slower than built-ins.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer uses a Python UDF to extract a value from a JSON string column. What is the most efficient replacement?</b><br>
A. A pandas UDF with <code>json.loads</code><br>
B. <code>from_json</code> (or <code>get_json_object</code>, or the <code>:</code> syntax in Databricks SQL)<br>
C. Collect the column to the driver and parse it<br>
D. Write the column to CSV and re-read it<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which approach computes each row's share of its group total with the fewest shuffles?</b><br>
A. <code>groupBy</code> + <code>join</code> back to the original rows<br>
B. A window function partitioned by the group key<br>
C. <code>orderBy</code> then <code>groupBy</code><br>
D. <code>repartition(1)</code> then a Python loop<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a UDF that normalizes product codes (<code>sku-in-00042</code> → <code>SKU-IN-00042</code>), and rewrite it with built-ins. Compare with <code>compare</code></li><li>Write a UDF that returns the number of item prices greater than the row's average item price, and rewrite it with higher-order functions</li><li>Rewrite this to use one shuffle: compute per-region max amount with <code>groupBy</code>, join it back, and flag rows equal to the max</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Normalize codes
lower = orders.select("order_id", F.lower("product_code").alias("code"))

@udf("string")
def normalize_udf(c):
    return c.upper().strip() if c else None

compare("normalize code", lower.select("order_id", normalize_udf("code").alias("c")),
        lower.select("order_id", F.upper(F.trim("code")).alias("c")))

# 2. Count prices above the row's average
@udf("int")
def above_avg_udf(prices):
    avg = sum(prices) / len(prices)
    return sum(1 for p in prices if p > avg)

avg_expr = F.aggregate("item_prices", F.lit(0.0), lambda a, p: a + p) / F.size("item_prices")
compare("above row average", orders.select("order_id", above_avg_udf("item_prices").alias("n")),
        orders.select("order_id", F.size(F.filter("item_prices", lambda p: p > avg_expr)).alias("n")))

# 3. One shuffle with a window
flagged = base.withColumn("is_max", F.col("amount") == F.max("amount").over(Window.partitionBy("region_code")))
print("shuffles:", shuffles(flagged))
flagged.filter("is_max").groupBy("region_code").count().show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Replace UDFs with built-ins: regex, JSON, maps or broadcast joins, date functions, higher-order functions</li><li>Verify rewrites on a sample: same results, then measure the speed-up</li><li>Use pandas UDFs only when a Python library is truly needed</li><li>Remove shuffles that don't change the result: redundant <code>repartition</code>, mid-pipeline <code>orderBy</code>, unnecessary <code>distinct</code></li><li>Add group values with window functions instead of <code>groupBy</code> + join back</li><li>Count <code>Exchange</code> operators to find shuffles</li></ul>
</div>

| Instead of | Use |
|---|---|
| Regex UDF | `F.regexp_extract`, `F.regexp_replace` |
| `json.loads` UDF | `F.from_json`, `F.get_json_object` |
| Dict lookup UDF | `F.create_map(...)[col]`, broadcast join |
| Loop over list | `F.transform`, `F.filter`, `F.aggregate` |
| groupBy + join back | `F.sum(...).over(Window.partitionBy(...))` |

## Git commit

```
git add 05_optimization/06_avoiding_udfs_and_shuffles.ipynb
git commit -m "add 05_06 avoiding udfs and shuffles notebook"
```